In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import scanpy as sc
import torch
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from sklearn.metrics.cluster import adjusted_rand_score
import SpaDiff as sd

In [ ]:
Batch_key="batch_name"
flag = True
st_samples = ["151673","151674","151675","151676"]
batch_list = []

In [ ]:
# 设置参数
parser = sd.get_args() 
args, unknown = parser.parse_known_args() 

args.n_clusters = 7
args.domains = 7

print(args.random_seed)
torch.manual_seed(args.random_seed)
torch.cuda.manual_seed_all(args.random_seed)
np.random.seed(args.random_seed)
random.seed(args.random_seed)
torch.backends.cudnn.deterministic = True

if args.cuda >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{args.cuda}")
    torch.cuda.manual_seed_all(args.random_seed)
    print(f"Using GPU: {torch.cuda.get_device_name(args.cuda)}")
else:
    device = torch.device("cpu")
    print("Using CPU.")

In [ ]:
for i,sample in enumerate(st_samples):  
    
    st_adata = sc.read_visium("../data/case1/"+sample+"/") 
    st_adata.var_names_make_unique()
    sc.pp.normalize_total(st_adata)
    sc.pp.log1p(st_adata)
    
    truth = pd.read_csv('../data/case1/'+sample+'/truth.txt', sep='\t', header=None, index_col=0)
    truth.columns = ['Truth']
    st_adata.obs['Truth'] = truth.loc[st_adata.obs_names, 'Truth']
    st_adata.obs["batch_name"] = sample
    st_adata.obs_names = [x+'-'+str(i) for x in st_adata.obs_names]

    # sc.pp.highly_variable_genes(st_adata, subset=True, flavor="seurat_v3", n_top_genes=3000)

    batch_list.append(st_adata)

adata = sc.concat(batch_list)

In [ ]:
adata

In [ ]:
sc.pp.highly_variable_genes(adata,subset=True, flavor="seurat_v3", n_top_genes=3000,batch_key="batch_name")
adata, HL = sd.spatial_rec_multi(adata,n_neighbors=args.n_neighbors,alpha=args.rec_alpha,)

In [ ]:
sc.pp.pca(adata, n_comps=50)
features = torch.tensor(adata.obsm['X_pca'].toarray() if issparse(adata.obsm['X_pca']) else np.array(adata.obsm['X_pca'])).float()

In [ ]:
clf=sd.DEC( features, HL, 
              node_width=features.shape[1],
              device = device,
              args=args)

a = clf.fit(features, HL)
y_pred, prob, z=clf.predict()
adata.obsm['z'] = z

In [ ]:
adata.obs['mclust'] = sd.utils.mclust_R(adata, used_obsm='z', pca_num = 30, num_cluster = args.domains)
adata.obs['mclust'] = adata.obs['mclust'].astype('int')
adata.obs['mclust'] = adata.obs['mclust'].astype('category')

In [ ]:
Batch_list = []
for sample in st_samples:
    Batch_list.append(adata[adata.obs['batch_name'] == sample])

In [ ]:
ARI_list = []
for i in range(len(st_samples)):
    obs_df = Batch_list[i].obs.dropna()
    ARI = adjusted_rand_score(obs_df["mclust"], obs_df['Truth'])
    ARI = round(ARI, 3)
    ARI_list.append(ARI)
print(ARI_list)

obs_df = adata.obs.dropna()
ARI = adjusted_rand_score(obs_df["mclust"], obs_df['Truth'])
ARI = round(ARI, 3)
print('Adjusted rand index of mclust = ', ARI)

In [ ]:
plot_color = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, ad in enumerate(Batch_list):
    sc.pl.spatial(
        ad,
        color="mclust",
        ax=axes[i],
        show=False,
        spot_size=120,
        palette=plot_color,
        legend_loc=None, 
        title=str(ARI_list[i])
    )
fig.suptitle(
    'ARI = %.3f' %ARI,
    fontsize=18,
    y=1.0
)
plt.tight_layout()
plt.show()

In [ ]:
sc.pp.neighbors(adata,use_rep='z')
sc.tl.umap(adata)
sc.pl.umap(adata, 
           color="batch_name", 
           size = 10,
           legend_fontsize=14,
           legend_fontoutline=4,
           )
sc.pl.umap(adata, 
           color="mclust", 
           size = 10,
           legend_fontsize=14,
           legend_fontoutline=4,
           )
sc.pl.umap(adata, 
           color="Truth", 
           size = 10,
           legend_fontsize=14,
           legend_fontoutline=4,
           )